In [20]:
import torch
from transformers import AutoModel, AutoTokenizer
import torch.nn as nn
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
from hasoc_model import *
%load_ext autoreload
%autoreload 2

In [3]:
df_clara = pd.read_csv("hasoc_dataset/train.tsv", sep="\t")
df_clara.columns = ["id", "text", "label_A", "label_B", "label_C"]
df_clara = df_clara[["text", "label_A", "label_B", "label_C"]] 
df_clara = encode_labels(df_clara)

In [31]:
df_claraA = df_clara.dropna(subset=["label_A_enc"])
labelsA_list = df_claraA["label_A_enc"].tolist()
df_claraA = df_claraA["text"].tolist()

df_claraB = df[df["label_A"] == "HOF"].dropna(subset=["label_B_enc"])
labelsB_list = df_claraB["label_B_enc"].tolist()
df_claraB = df_claraB["text"].tolist()

df_claraC = df[(df["label_A"] == "HOF") & (df["label_C"].isin(["UNT", "TIN"]))].dropna(subset=["label_C_enc"])
labelsC_list = df_claraC["label_C_enc"].tolist()
df_claraC = df_claraC["text"].tolist()

In [32]:
df_claraA[0:10]

["#DhoniKeepsTheGlove | WATCH: Sports Minister Kiren Rijiju issues statement backing MS Dhoni over 'Balidaan Badge', tells BCCI to take up the matter with ICC and keep government in the know as nation's pride is involved    https://t.co/zuo5335Rjr",
 '@politico No. We should remember very clearly that #Individual1 just admitted to treason . #TrumpIsATraitor  #McCainsAHero #JohnMcCainDay',
 '@cricketworldcup Guess who would be the winner of this #CWC19?     Team who gets maximum points from the abandoned matches 😄 #ShameOnICC #WIvsENG @ICC',
 "Corbyn is too politically intellectual for #BorisJohnsonShouldNotBePM   Can't wait   #GeneralElectionNow https://t.co/pt8KmjfxJj",
 'All the best to #TeamIndia for another swimming competition on Sunday against #Pakistan.     #INDvPAK #ShameOnICC  #CWC19 #CWC19Rains ☔☔ https://t.co/MG2cIE0zib',
 '@kellymiller513 @TheRealOJ32 I hope you remembered to wipe the blood off of you, after the pic was taken.  #bloodonhishands #murderer',
 '@ICC Latest des

In [33]:
sentences = [
    "I love all people, no matter where they come from.",
    "That group is disgusting and should not exist.",
    "We should build a more inclusive and respectful community.",
    "You don't belong here. Go back to your country."
]

In [34]:
class Paola(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", num_outputs=8, bin_outputs=5):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_outputs)
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, bin_outputs),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.regressor(pooled), self.classifier(pooled)

In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_paola = Paola().to(device)
model_paola.load_state_dict(torch.load("model2_loaded.pth", map_location=device))

print("model2_loaded.pth loaded and ready to use!")

model2_loaded.pth loaded and ready to use!


In [36]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
encodings = tokenizer(df_claraA[0:10], truncation=True, padding=True, max_length=128, return_tensors="pt")

model_paola.eval()
input_ids_paola = encodings['input_ids'].to(device)
attention_mask_paola = encodings['attention_mask'].to(device)

with torch.no_grad():
    preds_num, preds_bin = model_paola(input_ids=input_ids_paola, attention_mask=attention_mask_paola)

preds_num = preds_num.cpu().numpy()
preds_bin = preds_bin.cpu().numpy()
preds_bin = (preds_bin > 0.5).astype(int)

for idx, sentence in enumerate(df_claraA[0:5]):
    print(f"Sentence: {sentence}")
    print(f"Numerical predictions: {preds_num[idx]}")
    print(f"Binary predictions: {preds_bin[idx]}")
    print()

Sentence: #DhoniKeepsTheGlove | WATCH: Sports Minister Kiren Rijiju issues statement backing MS Dhoni over 'Balidaan Badge', tells BCCI to take up the matter with ICC and keep government in the know as nation's pride is involved    https://t.co/zuo5335Rjr
Numerical predictions: [1.4261322  1.026586   0.71562517 0.65788895 1.8180856  0.5279452
 1.3764592  0.03854378]
Binary predictions: [0 0 0 0 1]

Sentence: @politico No. We should remember very clearly that #Individual1 just admitted to treason . #TrumpIsATraitor  #McCainsAHero #JohnMcCainDay
Numerical predictions: [2.8258035  2.305355   1.8951546  1.7395177  2.1474316  1.3307736
 2.0716138  0.32836297]
Binary predictions: [0 0 0 0 0]

Sentence: @cricketworldcup Guess who would be the winner of this #CWC19?     Team who gets maximum points from the abandoned matches 😄 #ShameOnICC #WIvsENG @ICC
Numerical predictions: [2.2002656  1.9434769  1.5539635  1.3744228  2.0864556  0.9715351
 1.9307483  0.07670174]
Binary predictions: [1 0 0 0 0

numerical_cols = ['sentiment', 'respect', 'insult', 'humiliate', 'status',
                  'dehumanize', 'attack_defend', 'hatespeech']
                  
binary_cols = ['target_race', 'target_religion', 'target_origin', 'target_gender',
               'target_sexuality']

In [37]:
class ColineA(nn.Module):
    def __init__(self, task, model_name=None, num_labels=None, class_weights=None):
        super().__init__()
        self.task = task
        self.model_name = model_name or MODEL_NAMES[task]
        self.num_labels = num_labels or NUM_LABELS[task]
        self.class_weights = class_weights

        self.transformer = AutoModel.from_pretrained(self.model_name)
        hidden_size = self.transformer.config.hidden_size  # usually 768

        self.extra_feat_size = 13  # 8 numerical + 5 binary

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + self.extra_feat_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, self.num_labels)
        )

    def forward(self, input_ids, attention_mask, extra_features, labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # CLS token
        
        # Concatenate CLS embedding with extra features
        combined = torch.cat((pooled_output, extra_features), dim=1)

        logits = self.classifier(combined)

        if labels is not None:
            return {"logits": logits, "labels": labels}
        else:
            return {"logits": logits}


In [38]:
MODEL_NAMES = {
    "A": "roberta-base",
    "B": "GroNLP/hateBERT",
    "C": "GroNLP/hateBERT"
}
NUM_LABELS = {"A": 2, "B": 3, "C": 2}

In [39]:
task = "A"

In [40]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAMES[task], use_fast=True)
encodings = tokenizer(df_claraA[0:10], truncation=True, padding=True, return_tensors="pt")
input_ids_clara = encodings['input_ids'].to(device)
attention_mask_clara = encodings['attention_mask'].to(device)

In [41]:
extra_features = np.concatenate([preds_num, preds_bin], axis=1)
extra_features_tensor = torch.tensor(extra_features, dtype=torch.float32)

class_weights = compute_class_weights(labelsA_list[0:10], NUM_LABELS[task], task=task)

model_colineA = ColineA(task="A", model_name="roberta-base", class_weights=class_weights).to(device)
state_dict = torch.load("best_model_A_roberta-base.pth", map_location=device)

# Only load matching keys (transformer weights)
model_colineA.transformer.load_state_dict({
    k.replace("model.", ""): v for k, v in state_dict.items() if "transformer" in k or "model." in k
}, strict=False)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


_IncompatibleKeys(missing_keys=['embeddings.word_embeddings.weight', 'embeddings.position_embeddings.weight', 'embeddings.token_type_embeddings.weight', 'embeddings.LayerNorm.weight', 'embeddings.LayerNorm.bias', 'encoder.layer.0.attention.self.query.weight', 'encoder.layer.0.attention.self.query.bias', 'encoder.layer.0.attention.self.key.weight', 'encoder.layer.0.attention.self.key.bias', 'encoder.layer.0.attention.self.value.weight', 'encoder.layer.0.attention.self.value.bias', 'encoder.layer.0.attention.output.dense.weight', 'encoder.layer.0.attention.output.dense.bias', 'encoder.layer.0.attention.output.LayerNorm.weight', 'encoder.layer.0.attention.output.LayerNorm.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.output.dense.weight', 'encoder.layer.0.output.dense.bias', 'encoder.layer.0.output.LayerNorm.weight', 'encoder.layer.0.output.LayerNorm.bias', 'encoder.layer.1.attention.self.query.weight', 'encoder.layer.1.att

In [ ]:
dataset = Dataset.from_dict({
        "input_ids": input_ids_clara,
        "attention_mask": attention_mask_clara,
        "labels": torch.tensor(labelsA_list[0:10], dtype=torch.long).tolist(),
        "extra_features": extra_features_tensor.tolist()
    })

dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_model(task, model_colineA, dataset, tokenizer, resume=True)

⚠️ No checkpoint found — starting from scratch.


 25%|██▌       | 1/4 [28:47<1:26:21, 1727.08s/it]



                                          
                                             
 25%|██▌       | 1/4 [08:21<01:13, 24.54s/it]

{'loss': 0.2893, 'grad_norm': 1.745883822441101, 'learning_rate': 1.5000000000000002e-05, 'epoch': 1.0}


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (2, 2) + inhomogeneous part.